In [ ]:
%pip install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
import json
from google.colab import userdata

In [3]:
# Retrieve the secret by the name you gave it
groq_api_key = userdata.get('GROQ_API_KEY')

# Now you can use it in your logic
print("API Key loaded successfully!")

API Key loaded successfully!


In [4]:
loader = TextLoader("locations_and_compounds_data.txt")
docs = loader.load()  # Returns list of Document objects

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Max characters per chunk
    chunk_overlap=200,   # Overlap for context
    add_start_index=True # Track original position
)
chunks = splitter.split_documents(docs)

In [ ]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

In [6]:
# Persist to Chroma - best for local dev/production similarity search
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./locations_and_compounds_db"  # Saves locally
)

In [7]:
!zip -r locations_and_compounds_db.zip locations_and_compounds_db

  adding: locations_and_compounds_db/ (stored 0%)
  adding: locations_and_compounds_db/chroma.sqlite3 (deflated 51%)
  adding: locations_and_compounds_db/2dc2f681-6fd3-47b9-b57d-323232fe4ef8/ (stored 0%)
  adding: locations_and_compounds_db/2dc2f681-6fd3-47b9-b57d-323232fe4ef8/length.bin (deflated 70%)
  adding: locations_and_compounds_db/2dc2f681-6fd3-47b9-b57d-323232fe4ef8/index_metadata.pickle (deflated 45%)
  adding: locations_and_compounds_db/2dc2f681-6fd3-47b9-b57d-323232fe4ef8/link_lists.bin (deflated 78%)
  adding: locations_and_compounds_db/2dc2f681-6fd3-47b9-b57d-323232fe4ef8/header.bin (deflated 55%)
  adding: locations_and_compounds_db/2dc2f681-6fd3-47b9-b57d-323232fe4ef8/data_level0.bin (deflated 53%)


In [8]:
from google.colab import files
files.download("locations_and_compounds_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>